# Lecture 54 — Getting Started With MLflow

## Chapter-wise Zero-to-Hero Study Guide for a Senior SRE / DevOps Engineer

This notebook is prepared from the Udemy AI summary of the lecture **Getting Started With MLflow** from the course **Complete MLOps Bootcamp With 10+ End To End ML Projects**.

The purpose is not to repeat the summary. The purpose is to convert the lecture into a practical study guide that helps a Senior SRE understand MLflow as an operational platform component.

Use this notebook in two ways: read the theory sections first, then run the practice cells locally and inspect the MLflow UI.


# 1. What this chapter teaches

The lecture introduces MLflow by answering four questions: what MLflow is, why it is used, who uses it, and where it fits in real ML projects.

For a Senior SRE, the key message is this: MLflow is not just a data scientist notebook helper. It is a metadata and artifact control plane for machine learning workflows. It helps teams track experiments, compare model runs, package model artifacts, manage model versions, and eventually connect model development to production delivery.

The operational questions are familiar from DevOps: which exact input produced this artifact, who created it, where is it stored, can it be reproduced, can it be promoted safely, and can it be rolled back? MLflow gives ML teams a structure for answering these questions.


# 2. ML lifecycle from an SRE point of view

A normal service lifecycle starts from code, moves through CI, creates an artifact, deploys it, and then gets monitored. A machine learning lifecycle has all of that plus data preparation, exploratory data analysis, feature engineering, training, validation, model packaging, registry, serving, monitoring, and retraining.

The lecture mentions the lifecycle stages: data preparation, EDA, feature engineering, model training, validation, deployment, and monitoring. Each stage creates reliability risk. Data may change, feature logic may drift, dependencies may break, random seeds may make results non-reproducible, and a model may be deployed without a clear lineage.

MLflow helps by recording metadata around training and evaluation. It does not replace Kubernetes, CI/CD, Docker, DVC, Airflow, or observability. Instead, it becomes the ML-specific tracking layer that connects experiments and models to those systems.


# 3. DevOps-to-MLflow mental model

| DevOps / SRE concept | MLflow concept | Meaning |
|---|---|---|
| CI build | MLflow run | One execution of training or evaluation |
| Build metadata | Parameters and tags | Inputs and context of the run |
| Test results | Metrics | Accuracy, F1, RMSE, loss, latency, model quality |
| Build artifact | Model artifact | Model files, plots, reports, signatures |
| Artifact repository | Artifact store | Local folder, S3, GCS, Azure Blob, MinIO |
| Release candidate | Candidate model version | A model being considered for promotion |
| Production promotion | Model registry transition | Moving a model through approval lifecycle |
| Rollback | Revert model version | Return traffic to a previous known-good model |

A useful interview phrase: MLflow brings build traceability, artifact management, comparison, and promotion discipline to machine learning workflows.


# 4. Core components of MLflow

## MLflow Tracking

Tracking is the most important starting point. It records parameters, metrics, artifacts, tags, and run metadata. Parameters describe what you used, metrics describe how it performed, artifacts store the files produced, and tags make the run searchable and auditable.

## MLflow Projects

Projects describe a reproducible way to run ML code. For an SRE, think of this as a packaging convention for training jobs: what command to run, what parameters are accepted, and what environment is expected.

## MLflow Models

Models provide a standard packaging format across frameworks such as scikit-learn, TensorFlow, PyTorch, and custom Python. This helps reduce the 'works on my machine' handoff problem between data science and engineering.

## MLflow Model Registry

The registry manages model versions and lifecycle state. This becomes important when you need staging, production, archival, approval, and rollback semantics.


# 5. Why MLflow is used

The lecture highlights experiment tracking, code structuring, pipeline creation, model packaging, dependency management, hyperparameter evaluation, visual comparison of results, and collaboration through a shared UI.

From an SRE perspective, the strongest reason is operational traceability. Without MLflow, teams may save models as model.pkl, model_final.pkl, model_better.pkl, or model_really_final.pkl. That is not production-grade. A production ML platform needs to know which run generated the model, what parameters were used, what metrics were achieved, what artifacts were generated, and whether the model passed validation gates.

MLflow also makes hyperparameter tuning auditable. Hyperparameters are configuration values such as learning rate, max depth, solver, number of estimators, and random seed. Small changes can produce major behavior changes. Tracking them is similar to tracking deployment configuration for a critical service.


# 6. Who uses MLflow?

## Data engineers

Data engineers care about source data, ETL quality, schema changes, freshness, and feature availability. MLflow can record which dataset or feature source was used by a training run.

## Data scientists

Data scientists use MLflow to compare experiments, track parameters and metrics, save plots, and identify the best model candidate.

## ML engineers

ML engineers use MLflow to move from notebooks to reusable training code, model packaging, model serving, and deployment integration.

## MLOps / SRE engineers

SREs care about the reliability of the MLflow service, backend database, artifact store, network exposure, authentication, backup strategy, observability, and incident response. Once MLflow is shared by teams, it is a platform service and should be operated like one.


# 7. Production architecture mental model

For local learning, MLflow can store everything in a local mlruns folder. For production, separate the components:

```text
Notebook / training job
  -> MLflow Tracking Server
      -> backend store: PostgreSQL or MySQL
      -> artifact store: S3, GCS, Azure Blob, or MinIO
      -> model registry metadata in backend store
```

A production-style deployment should include TLS, private ingress or internal load balancer, SSO or authentication proxy, secrets management, database backups, object storage retention, monitoring, alerting, and runbook ownership.

Important SRE distinction: MLflow is usually not on the live inference hot path if models are already deployed. But it is critical for training, model registration, deployment workflow, auditability, and rollback.


# 8. Local practice setup

Run this from the repository MLflow folder:

```bash
cd /Users/jithinpjoseph/Documents/GitHub/SRE-Challenges/mlops/mlflow
python3 -m venv .venv
source .venv/bin/activate
pip install mlflow scikit-learn pandas matplotlib joblib
jupyter lab
```

After running the notebook, start the UI:

```bash
mlflow ui --backend-store-uri ./mlruns --host 127.0.0.1 --port 5000
```

Open `http://127.0.0.1:5000`.


In [ ]:
# Optional dependency installation. Run once if needed.
# !pip install -q mlflow scikit-learn pandas matplotlib joblib


In [ ]:
from pathlib import Path
import json
import platform
import sys

import mlflow
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print('Python:', sys.version)
print('Platform:', platform.platform())
print('MLflow version:', mlflow.__version__)


In [ ]:
tracking_dir = Path('mlruns').resolve()
mlflow.set_tracking_uri(tracking_dir.as_uri())
mlflow.set_experiment('lecture-54-getting-started-with-mlflow')

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment:', mlflow.get_experiment_by_name('lecture-54-getting-started-with-mlflow'))


# 9. Practice run 1: baseline model

This run trains a logistic regression model on the Iris dataset. The goal is not to master the algorithm here. The goal is to observe what MLflow captures: model type, solver, max iterations, random state, metrics, tags, plots, and the model artifact.


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape


In [ ]:
def log_confusion_matrix(y_true, y_pred, labels, artifact_path='plots'):
    fig, ax = plt.subplots(figsize=(6, 4))
    cm = confusion_matrix(y_true, y_pred)
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    display.plot(ax=ax)
    ax.set_title('Confusion Matrix')
    output_path = Path('confusion_matrix.png')
    fig.tight_layout()
    fig.savefig(output_path)
    plt.close(fig)
    mlflow.log_artifact(str(output_path), artifact_path=artifact_path)
    output_path.unlink(missing_ok=True)


In [ ]:
with mlflow.start_run(run_name='logistic-regression-baseline') as run:
    params = {
        'model_type': 'LogisticRegression',
        'solver': 'lbfgs',
        'max_iter': 200,
        'random_state': 42,
        'test_size': 0.25,
    }

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(solver=params['solver'], max_iter=params['max_iter'], random_state=params['random_state']))
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    metrics = {
        'accuracy': accuracy_score(y_test, predictions),
        'f1_macro': f1_score(y_test, predictions, average='macro'),
    }

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.set_tags({
        'team': 'sre-mlops-learning',
        'lecture': '54-getting-started-with-mlflow',
        'purpose': 'baseline-experiment',
        'owner_role': 'senior-sre',
        'python_version': platform.python_version(),
    })

    log_confusion_matrix(y_test, predictions, iris.target_names)
    mlflow.sklearn.log_model(pipeline, artifact_path='model', input_example=X_test.head(3))

    print('Run ID:', run.info.run_id)
    print('Metrics:', metrics)


# 10. Practice run 2: candidate model

Now train a second model and compare it. This simulates normal ML experimentation. In production, the comparison would become a quality gate before model registration or promotion.


In [ ]:
with mlflow.start_run(run_name='random-forest-candidate') as run:
    params = {
        'model_type': 'RandomForestClassifier',
        'n_estimators': 100,
        'max_depth': 3,
        'random_state': 42,
        'test_size': 0.25,
    }

    model = RandomForestClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        random_state=params['random_state'],
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    metrics = {
        'accuracy': accuracy_score(y_test, predictions),
        'f1_macro': f1_score(y_test, predictions, average='macro'),
    }

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.set_tags({
        'team': 'sre-mlops-learning',
        'lecture': '54-getting-started-with-mlflow',
        'purpose': 'candidate-experiment',
        'owner_role': 'senior-sre',
        'python_version': platform.python_version(),
    })

    log_confusion_matrix(y_test, predictions, iris.target_names)
    mlflow.sklearn.log_model(model, artifact_path='model', input_example=X_test.head(3))

    print('Run ID:', run.info.run_id)
    print('Metrics:', metrics)


# 11. Compare runs programmatically

The UI is helpful, but automation matters. A CI pipeline may need to decide whether a model is eligible for staging. The next cell searches runs, sorts them by F1 score, and applies a simple threshold.


In [ ]:
experiment = mlflow.get_experiment_by_name('lecture-54-getting-started-with-mlflow')
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.f1_macro DESC'],
)

columns_to_show = [
    'run_id',
    'tags.mlflow.runName',
    'params.model_type',
    'metrics.accuracy',
    'metrics.f1_macro',
    'status',
    'start_time',
]

runs_df[columns_to_show]


In [ ]:
MIN_F1_MACRO = 0.90
best_run = runs_df.sort_values('metrics.f1_macro', ascending=False).iloc[0]
best_run_name = best_run['tags.mlflow.runName']
best_f1 = best_run['metrics.f1_macro']
best_run_id = best_run['run_id']

print('Best run:', best_run_name)
print('Best run ID:', best_run_id)
print('Best f1_macro:', best_f1)

if best_f1 >= MIN_F1_MACRO:
    print('QUALITY GATE: PASS — candidate is eligible for further validation.')
else:
    print('QUALITY GATE: FAIL — do not promote this model.')


# 12. What to log in real projects

A production-grade ML run should log more than accuracy. Log parameters such as model type, learning rate, max depth, solver, random seed, dataset version, feature version, and preprocessing strategy. Log metrics such as precision, recall, F1, ROC AUC, RMSE, inference latency, model size, and business-specific quality indicators. Log artifacts such as trained models, validation reports, confusion matrices, feature importance plots, dependency lock files, sample payloads, and model cards. Log tags such as team, owner, git commit, CI build ID, branch, environment, Jira ticket, and risk classification.

Never log secrets as parameters or tags. API keys, database passwords, tokens, and private credentials must stay in secret managers.


In [ ]:
production_metadata = {
    'service_name': 'iris-classifier-demo',
    'team': 'sre-mlops-learning',
    'owner': 'senior-sre-practice',
    'training_pipeline': 'local-notebook',
    'git_commit': 'replace-with-real-commit-sha',
    'dataset': {
        'name': 'iris',
        'source': 'sklearn.datasets.load_iris',
        'version': 'built-in',
        'schema_fields': list(X.columns),
    },
    'quality_gates': {
        'min_f1_macro': MIN_F1_MACRO,
        'actual_f1_macro': float(best_f1),
        'status': 'pass' if best_f1 >= MIN_F1_MACRO else 'fail',
    },
}

metadata_path = Path('model_metadata.json')
metadata_path.write_text(json.dumps(production_metadata, indent=2))
print(metadata_path.read_text())


# 13. CI/CD integration pattern

A mature training pipeline can follow this flow: pull request merged, CI starts training, pipeline loads versioned data, model is trained, run is logged to MLflow, artifacts are stored, quality gates run, candidate is registered, staging deployment is triggered, smoke tests run, traffic is shifted, and monitoring decides whether to continue or roll back.

MLflow does not replace CI/CD. It gives CI/CD the model metadata and artifacts required to make safe promotion decisions.


In [ ]:
def decide_model_promotion(f1_score_value: float, min_threshold: float = 0.90) -> str:
    if f1_score_value >= min_threshold:
        return 'candidate_for_staging'
    return 'blocked_quality_gate_failed'

promotion_decision = decide_model_promotion(float(best_f1), MIN_F1_MACRO)
promotion_decision


# 14. Production checklist for SREs

When MLflow becomes a shared service, operate it like a platform dependency. Use a managed PostgreSQL or MySQL backend, durable object storage, TLS, authentication, secrets management, resource requests and limits, readiness and liveness checks, observability, backups, restore drills, and clear ownership.

Monitor request rate, error rate, latency, database connectivity, artifact upload/download failures, pod restarts, CPU, memory, disk usage if local storage exists, and failed training jobs caused by MLflow connectivity.

Security questions to ask: who can create experiments, who can read artifacts, who can approve models, who can promote to production, are artifacts encrypted, are credentials hidden, is the UI behind SSO, and are audit logs available?


# 15. Runbook: MLflow tracking server unavailable

## Symptoms

Data scientists cannot view experiments, training jobs fail while logging runs, model registration fails, deployment pipeline cannot resolve model versions, or MLflow UI returns 5xx/timeouts.

## First checks

```bash
kubectl get pods -n mlops
kubectl get svc -n mlops
kubectl get ingress -n mlops
kubectl logs deploy/mlflow-tracking-server -n mlops --tail=100
```

## Backend checks

Check PostgreSQL reachability, credentials, connection limits, migrations, disk, slow queries, and backup status.

## Artifact store checks

Check object storage permissions, IAM changes, bucket policy, network path, encryption policy, and storage errors.

## Mitigation

Restart unhealthy pods if safe, roll back recent changes, restore previous secret/config if credentials changed, pause model promotion if metadata integrity is uncertain, and communicate impact to ML teams.


# 16. Interview-ready explanation

MLflow is an open-source platform for managing the machine learning lifecycle. It helps teams track experiments, log parameters and metrics, store artifacts, package models, and manage model versions. From an SRE perspective, MLflow gives traceability and reproducibility to ML workflows, similar to how CI/CD and artifact repositories give traceability to software delivery.

A stronger Senior SRE answer: I would not run production MLflow as a local file-based setup. I would use a remote tracking server, a managed PostgreSQL backend store, durable object storage, IAM, TLS, SSO or reverse proxy authentication, backup and restore procedures, and dashboards for availability, latency, error rate, DB connections, and artifact failures. I would also connect MLflow metadata with Git commits, CI build IDs, dataset versions, and deployment events so that promotion and rollback are auditable.


# 17. Practice assignments

1. Add a third model such as SVC or KNeighborsClassifier and log it to MLflow.
2. Add Git metadata tags: commit SHA, branch, CI pipeline ID, author, and environment.
3. Create a model_card.md artifact with intended use, dataset, metrics, limitations, risk notes, and rollback plan.
4. Define an SLO for MLflow tracking availability, for example 99.5 percent monthly availability for the tracking API used by training and deployment pipelines.
5. Design a production MLflow architecture on Kubernetes with tracking server, PostgreSQL, object storage, ingress, authentication, observability, and backups.


In [ ]:
model_card = f'''# Model Card — Iris Classifier Demo

## Intended Use
Learning model for MLflow tracking practice from a Senior SRE perspective.

## Dataset
Iris dataset from scikit-learn.

## Selected Run
Run name: {best_run_name}
Run ID: {best_run_id}
F1 macro: {best_f1}

## Quality Gate
Minimum F1 macro: {MIN_F1_MACRO}
Decision: {promotion_decision}

## Operational Notes
For production, connect this to dataset versioning, model registry approval, canary deployment, inference monitoring, and rollback.
'''

Path('model_card.md').write_text(model_card)
print(model_card)


# 18. Final summary

This chapter establishes the foundation for MLflow. You learned that MLflow tracks experiments, parameters, metrics, artifacts, models, and model lifecycle metadata. The SRE takeaway is that MLflow introduces operational traceability into machine learning workflows.

When ML systems become production systems, traceability is not optional. You need to know what was trained, how it was trained, where it was stored, who approved it, how it was deployed, and how to roll back. That is why MLflow matters in MLOps.

Before the next lecture, make sure you can explain experiments, runs, parameters, metrics, artifacts, tracking server, backend store, artifact store, and why production MLflow should not rely only on local filesystem storage.
